In [ ]:
# Load the required packages
import scanpy as sc
import scvi
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import anndata as ad
import time
import tracemalloc
import os
import psutil

In [ ]:
# Set up the memory tracker
process = psutil.Process(os.getpid())

def rss_mb():
    return process.memory_info().rss / 1024 / 1024

In [4]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.3.1.post1


In [ ]:
# Load data 
print("RSS before:", rss_mb(), "MB")
adata = sc.read_h5ad("./prep_FINAL.h5ad")
print("RSS after:", rss_mb(), "MB")

RSS before: 163.40625 MB
RSS after: 436.125 MB


In [ ]:
# Converting the expression matrix to the Compressed Sparse Row format for more efficient processing
adata.X = adata.X.tocsr()
adata.layers["counts"] = adata.X.copy()

## Split the object by subject/individual

In [ ]:
# 1. Split the anndata object into a disctionary of objects based on the individual id
split_column = 'new_id'
categories = adata.obs[split_column].unique()

# Create a dictionary to hold the split AnnData objects
split_adata_dict = {
    category: adata[adata.obs[split_column] == category, :].copy()
    for category in categories
}

print(f"Split AnnData into {len(split_adata_dict)} objects.")


In [ ]:
split_adata_dict
print("RSS after:", rss_mb(), "MB")

RSS after: 1702.9375 MB


# Batch correction with SCVI by subject/individual

In [ ]:
# Set up the early stopping criteria based on the developer's reccomendations 
early_stopping_kwargs = {
    "early_stopping": True,
    "early_stopping_monitor": "elbo_validation",
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
    "check_val_every_n_epoch": 1,
}

In [ ]:
# --- 2. Loop through and perform operations ---

start = time.perf_counter() # track time
tracemalloc.start() # track memory
print("RSS before:", rss_mb(), "MB")

# Loop through each category and its corresponding AnnData object
for category, subset_adata in split_adata_dict.items():
    print(f"\nProcessing subset for category: '{category}'")
    
    # Perform an scvi model training 
    scvi.model.SCVI.setup_anndata(subset_adata, layer="counts", batch_key="assay")
    
    # Now, we are setting up the parameters of the model. They are non-default but have been verified to generally work well in the integration task
    model = scvi.model.SCVI(subset_adata, n_layers=2, n_latent=30, gene_likelihood="nb")
    
    # Train the model
    model.train(accelerator = "mps", **early_stopping_kwargs)

    # Predict as if 5′
    predicted_5prime_expr = model.get_normalized_expression(
    subset_adata,
    transform_batch="10x 5' v1",
    library_size=10000 
    )

    end = time.perf_counter()
    print(f"Total loop runtime: {end - start:.4f} seconds")
    current, peak = tracemalloc.get_traced_memory()
    print(f"Peak memory during loop: {peak / 1024 / 1024:.2f} MB")
    print("RSS after:", rss_mb(), "MB")
    
    # Store the modified AnnData object in a new dictionary
    subset_adata.layers["predicted"] = predicted_5prime_expr
    split_adata_dict[category] = subset_adata

print(f"Peak memory after loop: {peak / 1024 / 1024:.2f} MB")
tracemalloc.stop()

In [ ]:
# --- 3. Concatenate the modified objects back together ---
merged_adata = ad.concat(
    split_adata_dict,
    join='outer',
    label='original_batch',
    fill_value=0
)


In [ ]:
# Save the dataset
merged_adata.write("./DS_scvi.h5ad") # Specify the dataset

In [ ]:
## OR for the UseCase
merged_adata.write("./DS_merged_scvi_25.h5ad")